# 22 · Alignment, Post-Training and Self-Supervised Learning

- **Level:** Modern (2018 → present)
- **Typical semester:** research and self-study
- **Prerequisites:** 14, 19, 21
- **Matches code in:** `lipipy.research` (alignment) and `lipipy.dl` (self-supervised)


## 1. Supervised fine-tuning (SFT)

$\mathcal L_{SFT} = -\sum_{t\in\text{response}}\log\pi_\theta(y_t|x, y_{<t})$, with the loss masked on prompt tokens.


## 2. Reward modelling

**Bradley-Terry preference model:** $P(y_w\succ y_l|x) = \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))$

Reward loss: $\mathcal L_R = -\mathbb E\left[\log\sigma(r_\phi(x,y_w) - r_\phi(x, y_l))\right]$

For $K$ ranked responses: Plackett-Luce $P(\tau) = \prod_k\frac{e^{r(y_{\tau_k})}}{\sum_{j\ge k}e^{r(y_{\tau_j})}}$


## 3. RLHF with PPO

Per-token reward $r_t = -\beta\log\frac{\pi_\theta(y_t|\cdot)}{\pi_{ref}(y_t|\cdot)}$, plus $r_\phi(x, y)$ at the final token. Optimize with PPO-clip (see 21 §8) and a value head for GAE.

Objective: $\max_\pi\mathbb E[r_\phi(x,y)] - \beta D_{KL}(\pi\|\pi_{ref})$


## 4. Direct preference optimization (DPO) and relatives

From the closed-form optimum (21 §13), $r(x,y) = \beta\log\frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta\log Z(x)$. Plugging into Bradley-Terry cancels $Z$:
$$\mathcal L_{DPO} = -\mathbb E\left[\log\sigma\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right]$$

**IPO:** $\left(h_\theta - \frac{1}{2\beta}\right)^2$ where $h_\theta$ is the log-ratio difference (avoids overfitting to deterministic preferences)

**KTO:** uses unpaired good/bad labels with a prospect-theory value function $v = \sigma(\beta(r_\theta - z_0))$ for desirable, $\sigma(\beta(z_0 - r_\theta))$ for undesirable outputs

**ORPO:** $\mathcal L_{SFT} + \lambda\left(-\log\sigma\left(\log\frac{\text{odds}_\theta(y_w)}{\text{odds}_\theta(y_l)}\right)\right)$, $\text{odds}(y) = \frac{p(y)}{1 - p(y)}$ (no reference model)

**SimPO:** reference-free, length-normalized reward $\frac{\beta}{|y|}\log\pi_\theta(y|x)$ with a target margin $\gamma$


## 5. Group relative policy optimization (GRPO) and RL with verifiable rewards

Sample $G$ responses per prompt, with rewards $r_1,\dots,r_G$ (often from a verifier: correct answer, passing tests). Group-normalized advantage:
$$\hat A_i = \frac{r_i - \text{mean}(r_{1:G})}{\text{std}(r_{1:G})}$$
$$\mathcal J_{GRPO} = \mathbb E\left[\frac1G\sum_i\frac{1}{|y_i|}\sum_t\min\left(\rho_{i,t}\hat A_i, \text{clip}(\rho_{i,t}, 1 - \epsilon, 1 + \epsilon)\hat A_i\right) - \beta D_{KL}(\pi_\theta\|\pi_{ref})\right]$$
No value network is needed. Variants change the normalization (e.g. drop the std or length terms) or the clipping range.

**Rejection-sampling fine-tuning / best-of-$N$:** sample $N$, keep the highest-reward ones, run SFT on them. Best-of-$N$ vs base KL $\approx\log N - \frac{N-1}{N}$.

**Process reward models:** score each reasoning step; outcome reward models score only the final answer.


## 6. Constitutional AI / RLAIF

Replace human preference labels with model-generated critiques and preferences guided by a written set of principles, then train with the same reward-model or DPO machinery.


## 7. Reward hacking and over-optimization

Goodhart's law: optimizing a proxy reward $\hat r$ eventually decreases the true reward. Empirically the gold reward follows $d(\alpha - \beta d)$ in $d = \sqrt{KL(\pi\|\pi_{ref})}$ for best-of-$N$, rising then falling. KL regularization, reward-model ensembles and early stopping limit it.


## 8. Contrastive self-supervised learning

**InfoNCE / NT-Xent (SimCLR):**
$$\ell_{i} = -\log\frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k\ne i}\exp(\text{sim}(z_i,z_k)/\tau)}, \qquad \text{sim}(u,v) = \frac{u^\top v}{\|u\|\|v\|}$$

**CLIP:** symmetric cross-entropy over the $N\times N$ image-text similarity matrix $S_{ij} = \frac{\langle f(I_i), g(T_j)\rangle}{\tau}$, with labels on the diagonal: $\mathcal L = \frac12[CE(S, \text{diag}) + CE(S^\top, \text{diag})]$. **SigLIP:** a per-pair sigmoid loss $-\log\sigma(z_{ij}(tS_{ij} + b))$, $z_{ij} = \pm1$.

**MoCo:** queue of negatives + momentum encoder $\theta_k\leftarrow m\theta_k + (1 - m)\theta_q$


## 9. Non-contrastive self-supervised learning

**BYOL:** $\left\|\bar q_\theta(z_\theta) - \bar z'_\xi\right\|^2$ with a predictor head and EMA target $\xi\leftarrow\tau\xi + (1 - \tau)\theta$. **SimSiam:** same idea with stop-gradient instead of EMA.

**Barlow Twins:** $\sum_i(1 - C_{ii})^2 + \lambda\sum_i\sum_{j\ne i}C_{ij}^2$, $C$ = cross-correlation matrix of the two views' embeddings

**VICReg:** invariance (MSE) + variance ($\max(0, \gamma - \text{std}(z_j))$) + covariance (off-diagonal squared) terms

**DINO:** cross-entropy between a sharpened, centred teacher softmax and the student softmax: $-\sum P_t(x)\log P_s(x')$, $P_t = \text{softmax}((g_t(x) - c)/\tau_t)$


## 10. Masked and predictive self-supervision

**MAE:** mask ~75% of image patches, reconstruct pixels with MSE on the masked patches only.

**JEPA (I-JEPA, V-JEPA):** predict the *embeddings* of masked target blocks from a context block, $\sum_{i}\|\hat s_{y_i} - s_{y_i}\|^2$, with an EMA target encoder. No pixel reconstruction.

**Data2vec:** regress the teacher's averaged top-$K$ layer representations of masked inputs.


## 11. Alignment evaluation and safety

Win rate against a reference with a judge model; length-controlled win rate; Elo / Bradley-Terry ratings from pairwise comparisons, $P(A\text{ beats }B) = \frac{1}{1 + 10^{(R_B - R_A)/400}}$; Elo update $R_A\leftarrow R_A + K(S_A - E_A)$.

Calibration of model confidence (ECE, see 17 §16); refusal and jailbreak rates; prompt-injection test suites.
